# Kev-9B: best.pt of every paper run on Grummon 2025, Kreps 2020 (COVID), Kantorowicz 2026, and Magistro 2025

What and why: `README.md` in this folder.

In [ ]:
# Environment flags, set before any Hugging Face or torch import (same as slurm/train-notebook.sbatch)
import os
from pathlib import Path

# On a cluster (Slurm job, incl. JupyterLab sessions), use the shared Hugging Face cache on the project space
ON_CLUSTER = "SLURM_JOB_ID" in os.environ
CLUSTER_HF_HOME = Path.home() / "links" / "projects" / "def-yuma" / "wadi" / "hf"
if ON_CLUSTER:
    if not CLUSTER_HF_HOME.is_dir():
        raise FileNotFoundError(f"Cluster Hugging Face cache not found: {CLUSTER_HF_HOME}")
    os.environ["HF_HOME"] = str(CLUSTER_HF_HOME)

# Never download: the base model must already be in the Hugging Face cache
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# Lets the CUDA allocator grow segments instead of fragmenting
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(f"ON_CLUSTER: {ON_CLUSTER}")
if ON_CLUSTER:
    print(f"HF_HOME: {os.environ['HF_HOME']}")

In [9]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

# Walk up from the notebook folder to locate the repository's `code/` root
CODE_DIR = next(
    (c for p in (Path.cwd(), *Path.cwd().parents) for c in (p, p / "code") if all((c / d).is_dir() for d in ("kev", "data", "models"))),
    None,
)
if CODE_DIR is None:
    raise RuntimeError(f"No code/ root containing kev/, data/, and models/ at or above {Path.cwd()}")

PAPER_DIR = CODE_DIR / "papers" / "beta-loss"
SRC_DIR = PAPER_DIR / "src"

# Run from `code/` so data loaders and shared disk caches resolve identically to the training notebooks
os.chdir(CODE_DIR)
for _p in (str(CODE_DIR), str(CODE_DIR / "kev"), str(SRC_DIR)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from models.kev_pointer import KevPointerModel
from training import free_gpu

import beta_ols
from beta_train import runs as run_store
from beta_train.model_state import reset_trainable, set_checkpointing, snapshot_trainable
from beta_train.progress import PROGRESS_INTERVAL_SEC, HtmlProgress
from beta_train.validation import PopulationValidator, validate_all, validate_all_cached, validation_metrics_frame

NOTEBOOK = "04-test-best-9b"
CACHE_DIR = CODE_DIR / ".cache" / "beta_training"
MODEL_SLUG = "kev-9b"
# Runs pulled by pull-rorqual-runs.sh (Rows 2 and 2 ablation, trained on Rorqual) and pull-nibi-runs.sh
# (Rows 2b to 5, trained on Nibi): run files + checkpoints/best.pt
RUNS_DIRS = [
    PAPER_DIR / "runs-rorqual" / MODEL_SLUG,
    PAPER_DIR / "runs-nibi" / MODEL_SLUG,
]
# Each run's test results go in <run folder>/test_best/
TEST_FOLDER_NAME = "test_best"

# Every run folder (a folder with run.json) of both runs folders
RUN_DIRS = []
for runs_dir in RUNS_DIRS:
    for run_dir in sorted(runs_dir.iterdir()):
        if (run_dir / "run.json").exists():
            RUN_DIRS.append(run_dir)

# One run name in both runs folders would be two copies of the same regime: stop instead of testing either
seen_run_names = []
for run_dir in RUN_DIRS:
    if run_dir.name in seen_run_names:
        raise ValueError(f"{run_dir.name} is in more than one of {RUNS_DIRS}")
    seen_run_names.append(run_dir.name)

for run_dir in RUN_DIRS:
    print(run_dir.relative_to(PAPER_DIR))

# Which runs this notebook tests: "all" (every run, one after another) or one run folder name. Row 1 is not tested
# here: it belongs to the sim-respondents paper, whose test_best/ results this paper reads.
# A batch job sets it with the TEST_RUN environment variable (slurm/test-best.sbatch), so
# several jobs can test different runs at the same time; in JupyterLab, keep "all" or set it here.
TEST_RUN = "all"
if "TEST_RUN" in os.environ:
    TEST_RUN = os.environ["TEST_RUN"]
RUN_NAMES = [run_dir.name for run_dir in RUN_DIRS]
if TEST_RUN == "all":
    SELECTED_RUN_DIRS = RUN_DIRS
elif TEST_RUN in RUN_NAMES:
    SELECTED_RUN_DIRS = [RUN_DIRS[RUN_NAMES.index(TEST_RUN)]]
else:
    raise ValueError(f"TEST_RUN={TEST_RUN!r} is not 'all' or one of {RUN_NAMES}")
print(f"TEST_RUN: {TEST_RUN} (runs: {[run_dir.name for run_dir in SELECTED_RUN_DIRS]})")


runs-rorqual/kev-9b/row2_ablation_wu_table_no_kl
runs-rorqual/kev-9b/row2_wu_table_anchored
runs-nibi/kev-9b/row2b_wu_ce_oracle
runs-nibi/kev-9b/row3_berke_ce_only
runs-nibi/kev-9b/row4_hybrid_berke_ce_wu_table_anchored
runs-nibi/kev-9b/row4_hybrid_berke_ce_wu_table_no_kl
runs-nibi/kev-9b/row5_full_berke_wu_ce_oracle


In [2]:
# Same backbone and LoRA shape as the training notebooks, so best.pt loads into it
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required")
device = "cuda"
free_gpu()

KEV_BASE = "jaredpalmer/kev-9b@db029f08b290afd9fee4aa4bbcd9ae48602d1eb0"
LORA_RANK = 64
WEIGHTS_DTYPE = "bf16"
HEAD_TEMPERATURE = 1.0

kev_model = KevPointerModel.build(
    KEV_BASE,
    device=device,
    lora_rank=LORA_RANK,
    weights_dtype=WEIGHTS_DTYPE,
)
kev_model.net.head.temperature = HEAD_TEMPERATURE
set_checkpointing(kev_model, False)

INIT_TRAINABLE_STATE = snapshot_trainable(kev_model)
# Same key as the training notebooks, so their cached step-0 validations are reused
MODEL_KEY = {
    "base": KEV_BASE,
    "lora_rank": LORA_RANK,
    "weights_dtype": WEIGHTS_DTYPE,
    "head_temperature": HEAD_TEMPERATURE,
}
print(f"{KEV_BASE}: {len(INIT_TRAINABLE_STATE)} trainable tensors")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

jaredpalmer/kev-9b@db029f08b290afd9fee4aa4bbcd9ae48602d1eb0: 500 trainable tensors


## Test datasets

In [10]:
TEST_DATASETS = ["grummon2025", "covid", "kantorowicz2026", "magistro2025"]
VALIDATION_MAX_QUESTIONS = 1024
VALIDATION_MAX_CHARS = 1_000_000

validators = {}
for dataset_name in TEST_DATASETS:
    dataset_module = beta_ols.DATASET_MODULES[dataset_name]
    choice_df = dataset_module.load_choice_df()
    rows = dataset_module.regression_rows(choice_df)
    specs = dataset_module.regression_specs(rows)
    validators[dataset_name] = PopulationValidator(
        dataset_module,
        choice_df,
        specs,
        max_questions=VALIDATION_MAX_QUESTIONS,
        max_chars=VALIDATION_MAX_CHARS,
    )
display(pd.DataFrame([v.describe() for v in validators.values()]).set_index("validator"))


# Writes metrics.csv (one row per test dataset), betas.csv (model vs human betas, stacked) and test.json (what was tested)
def save_test_results(test_dir: Path, validation: dict, test_info: dict) -> None:
    test_dir.mkdir(parents=True, exist_ok=True)
    metrics_df = validation_metrics_frame(validation).T
    metrics_df.index.name = "dataset"
    run_store.write_text_atomic(test_dir / "metrics.csv", metrics_df.to_csv())
    run_store.write_betas_file(test_dir / "betas.csv", validation)
    run_store.write_text_atomic(test_dir / "test.json", json.dumps(test_info, indent=2))


# Saves every dataset in `results`: the TEST_DATASETS finished so far (in TEST_DATASETS order), then any earlier-saved
# dataset no longer in TEST_DATASETS (kept, never deleted). test.json lists exactly these, so a rerun reuses them and
# evaluates the rest. Returns the saved results.
def save_merged_test_results(test_dir: Path, results: dict, test_info: dict) -> dict:
    current = [d for d in TEST_DATASETS if d in results]
    kept = [d for d in results if d not in TEST_DATASETS]
    merged = {d: results[d] for d in current + kept}
    merged_info = dict(test_info)
    merged_info["test_datasets"] = list(merged)
    save_test_results(test_dir, merged, merged_info)
    return merged


# True when test_dir already holds results for this checkpoint step and model config on every dataset in TEST_DATASETS
# (other saved datasets do not matter)
def already_tested(test_dir: Path, checkpoint_step: int) -> bool:
    test_json = test_dir / "test.json"
    if not test_json.exists():
        return False
    stored = json.loads(test_json.read_text())
    if stored["checkpoint_step"] != checkpoint_step or stored["model"] != MODEL_KEY:
        return False
    missing = [d for d in TEST_DATASETS if d not in stored["test_datasets"]]
    return len(missing) == 0


# Loads every already-saved dataset result when test_dir matches this checkpoint step and model config, so adding a
# new dataset to TEST_DATASETS evaluates only the missing dataset(s) and merges cleanly (datasets no longer in
# TEST_DATASETS are loaded too, so they are kept on the next save).
# test.json is written last, so it is the record of what is complete: metrics.csv and betas.csv must hold every
# dataset it lists; an extra dataset in them (a save stopped before test.json was written) is ignored.
def load_existing_test_results(
    test_dir: Path, checkpoint_step: int
) -> dict[str, tuple[dict[str, float], pd.DataFrame]]:
    test_json = test_dir / "test.json"
    if not test_json.exists():
        return {}
    metrics_csv = test_dir / "metrics.csv"
    betas_csv = test_dir / "betas.csv"
    if not metrics_csv.exists() or not betas_csv.exists():
        raise FileNotFoundError(f"Incomplete test_best directory in {test_dir}")
    stored = json.loads(test_json.read_text())
    if stored["checkpoint_step"] != checkpoint_step or stored["model"] != MODEL_KEY:
        return {}
    stored_datasets = list(stored["test_datasets"])
    metrics_df = pd.read_csv(metrics_csv, float_precision="round_trip").set_index("dataset")
    beta_dfs = run_store.read_betas_file(betas_csv)
    not_in_metrics = [ds for ds in stored_datasets if ds not in metrics_df.index]
    not_in_betas = [ds for ds in stored_datasets if ds not in beta_dfs]
    if not_in_metrics or not_in_betas:
        raise ValueError(
            f"Inconsistent test_best artifacts in {test_dir}: test.json lists {stored_datasets}, "
            f"but metrics.csv lacks {not_in_metrics} and betas.csv lacks {not_in_betas}"
        )
    return {
        ds: ({k: float(v) for k, v in metrics_df.loc[ds].items()}, beta_dfs[ds])
        for ds in stored_datasets
    }


,kind,tasks,packed_persona_states,regressions,max_questions,max_chars,graph_limits,prep_sec
validator,,,,,,,,
grummon2025,fused,33798,4906,grummon2025,1024,1000000,"BANK_WIDTH=512, GRAPH_STATE=512, GRAPH_STATES=...",6.592
covid,fused,9855,1845,covid,1024,1000000,"BANK_WIDTH=512, GRAPH_STATE=512, GRAPH_STATES=...",1.959


## best.pt of every finished run

In [27]:
# best.pt of every finished run in SELECTED_RUN_DIRS (TEST_RUN); unfinished runs are skipped (rerun this notebook once they finish)
progress_bar = HtmlProgress(
    total=0, initial=0, description="test (waiting)", unit="batch", min_interval_sec=PROGRESS_INTERVAL_SEC
)
for run_dir in SELECTED_RUN_DIRS:
    run_info = run_store.read_run_json(run_dir)
    row_code = run_info["identity"]["run_metadata"]["paper_row"]
    if run_info["status"] != "finished":
        print(f"{row_code} {run_dir.name}: {run_info['status']}, skipped")
        continue
    test_dir = run_dir / TEST_FOLDER_NAME
    checkpoint = run_store.load_best(run_dir)
    best = checkpoint["best"]
    checkpoint_step = int(best["step"])
    if already_tested(test_dir, checkpoint_step):
        print(f"{row_code} {run_dir.name}: best.pt (step {checkpoint_step}) already tested, skipped")
        continue
    existing = load_existing_test_results(test_dir, checkpoint_step)
    missing_datasets = [d for d in TEST_DATASETS if d not in existing]
    test_info = {
        "notebook": NOTEBOOK,
        "model": MODEL_KEY,
        "row": row_code,
        "checkpoint": "best.pt",
        "checkpoint_step": checkpoint_step,
        "best_metric": best["metric"],
        "best_value": float(best["value"]),
        "test_datasets": TEST_DATASETS,
    }
    # One missing dataset at a time, saved as soon as it finishes (a stopped job keeps the datasets it finished)
    results = dict(existing)
    if missing_datasets:
        reset_trainable(kev_model, checkpoint["trainable"])
    for dataset_name in missing_datasets:
        new_result = validate_all(kev_model, {dataset_name: validators[dataset_name]}, progress_bar)
        results[dataset_name] = new_result[dataset_name]
        save_merged_test_results(test_dir, results, test_info)
    validation = save_merged_test_results(test_dir, results, test_info)
    reused = [d for d in TEST_DATASETS if d not in missing_datasets]
    note = f"tested {missing_datasets}, reused {reused}" if reused else f"tested {missing_datasets}"
    print(f"{row_code} {run_dir.name}: best.pt (step {checkpoint_step}) {note}")
    display(validation_metrics_frame(validation).round(5))
progress_bar.close()
reset_trainable(kev_model, INIT_TRAINABLE_STATE)


Row 2 (Ablation) row2_ablation_wu_table_no_kl: best.pt (step 320) already tested, skipped
Row 2 row2_wu_table_anchored: best.pt (step 320) already tested, skipped
Row 2b row2b_wu_ce_oracle: best.pt (step 928) already tested, skipped
Row 3 row3_berke_ce_only: best.pt (step 256) already tested, skipped
Row 4 row4_hybrid_berke_ce_wu_table_anchored: best.pt (step 1504) already tested, skipped
Row 4 (Ablation) row4_hybrid_berke_ce_wu_table_no_kl: best.pt (step 2240) tested


,grummon2025,covid
kl_loss,0.82268,1.11061
beta_mse,0.00610,0.00718
beta_mae,0.07619,0.06232
beta_mape_pct,69.50128,122.88789
spearman_rho,0.90000,0.95455
eval_prep_sec,0.02502,0.00943
eval_gpu_sec,552.41716,246.70584
eval_wall_sec,556.07227,250.51943
eval_ms_per_task,16.45282,25.42054


Row 5 row5_full_berke_wu_ce_oracle: best.pt (step 1600) already tested, skipped


## Summary

In [ ]:
# Every tested checkpoint of this model: one row per (row, run folder, test dataset)
test_jsons = []
for runs_dir in RUNS_DIRS:
    test_jsons.extend(sorted(runs_dir.glob(f"*/{TEST_FOLDER_NAME}/test.json")))
summary_frames = []
for test_json in test_jsons:
    test_dir = test_json.parent
    test_info = json.loads(test_json.read_text())
    metrics_df = pd.read_csv(test_dir / "metrics.csv")
    metrics_df.insert(0, "checkpoint_step", test_info["checkpoint_step"])
    metrics_df.insert(0, "slug", test_dir.parent.name)
    metrics_df.insert(0, "row", test_info["row"])
    summary_frames.append(metrics_df)
summary_df = pd.concat(summary_frames, ignore_index=True)
summary_df = summary_df.sort_values(["dataset", "row", "slug"]).set_index(["dataset", "row", "slug"])
display(summary_df[["checkpoint_step", "beta_mse", "beta_mae", "spearman_rho", "kl_loss"]].round(5))
